In [ ]:
import pandas as pd 
from pandas import DataFrame
from nltk.corpus import wordnet as wn
from collections import defaultdict
import re
from fuzzywuzzy import fuzz

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import json
import operator as op
import simplejson as json 

In [ ]:
concept_a_verifie = pd.read_csv('../Data_extracted/data_using_sim_sif_with_concept_a_verifie.csv', error_bad_lines= False, encoding='utf-8', names = ["Name_MeSH", "def_MeSH", "word_list_def_MeSH", "Name_WN", "def_WN", "word_list_def_WN", "Similarity_highest"])


In [ ]:
concept_a_verifie.head()

In [ ]:
concept_a_verifie.shape[0]

In [ ]:
dataConceptMeSH = pd.read_csv('../Data_extracted/data_complet_MeSH(V3).csv', error_bad_lines= False, encoding='utf-8', names = ["c_ui", "d_name", "d_ui", "rel", "scopeNote", "tree", "word_list"])

In [ ]:
dataConceptMeSH.head()

In [ ]:
dataConceptMeSH.shape[0]

In [ ]:
dataConceptWN = pd.read_csv('../Data_extracted/data_complet_WN(v3).csv', error_bad_lines= False, encoding='utf-8', names = ["c_name", "synset", "hyponymes", "hypernyms", "definition", "exemple", "list_word"])

In [ ]:
dataConceptWN.head()

In [ ]:
dataConceptWN.shape[0]

In [ ]:
#---------- Classifie all concept WN and MeSH by classe alphabet 

In [ ]:
def classifie_concept_WN_to_dic(dataConceptWN):
    keys_alphabet = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z','other']
    dic_alphabet_WN = defaultdict(str)
    dic_alphabet_WN = {key:[] for key in keys_alphabet}
    #print(dic_alphabet_WN)
    #dic_alphabet_WN[dataConceptWN.iloc[4].c_name[0]].append("aze")
    #dic_alphabet_WN[dataConceptWN.iloc[4].c_name[0]].append("aze")
    #print(dic_alphabet_WN)
    for i in range(dataConceptWN.shape[0]):
        # Using frist alphabet of concept to index dectionnaire and collect concept of this alphabet.
        try :
            dataConceptWN.iloc[i].c_name[0].lower()
        except Exception:
            dic_alphabet_WN['other'].append(dataConceptWN.iloc[i])
        else:
            if dataConceptWN.iloc[i].c_name[0].lower() in keys_alphabet:
                frist_alphabet = dataConceptWN.iloc[i].c_name[0].lower()
                dic_alphabet_WN[frist_alphabet].append(dataConceptWN.iloc[i])
            else:
                dic_alphabet_WN['other'].append(dataConceptWN.iloc[i])
    return dic_alphabet_WN

In [ ]:
dic_alphabet_WN = classifie_concept_WN_to_dic(dataConceptWN)

In [ ]:
# Other type concept, for exemple : int, float.
print(len(dic_alphabet_WN['a']))
print(len(dic_alphabet_WN['b']))
print(len(dic_alphabet_WN['other']))
#dic_alphabet_WN['other']

In [ ]:
def classifie_concept_MeSH_to_dic(dataConceptMeSH):
    keys_alphabet = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z','other']
    dic_alphabet_MeSH = defaultdict(str)
    dic_alphabet_MeSH = {key:[] for key in keys_alphabet}
    for i in range(dataConceptMeSH.shape[0]):
        # Using frist alphabet of concept to index dectionnaire and collect concept of this alphabet.
        try :
            dataConceptMeSH.iloc[i].d_name[0].lower()
        except Exception:
            dic_alphabet_MeSH['other'].append(dataConceptMeSH.iloc[i])
        else:
            if dataConceptMeSH.iloc[i].d_name[0].lower() in keys_alphabet:
                frist_alphabet = dataConceptMeSH.iloc[i].d_name[0].lower()
                dic_alphabet_MeSH[frist_alphabet].append(dataConceptMeSH.iloc[i])
            else:
                dic_alphabet_MeSH['other'].append(dataConceptMeSH.iloc[i])
    return dic_alphabet_MeSH


In [ ]:
dic_alphabet_MeSH = classifie_concept_MeSH_to_dic(dataConceptMeSH)

In [ ]:
# Other type concept, for exemple : int, float.
print(len(dic_alphabet_MeSH['a']))
print(len(dic_alphabet_MeSH['b']))
print(len(dic_alphabet_MeSH['other']))
#dic_alphabet_MeSH['other'][:10]

In [ ]:
def find_concept_in_MeSH(identifiant_concept, dic_alphabet_MeSH):
    try:
        identifiant_concept[0].lower()
    except Exception:
        return "Not found"
    else:
        classe_in_dic = dic_alphabet_MeSH[identifiant_concept[0].lower()]
        for i in range(len(classe_in_dic)):
            if(op.eq(identifiant_concept, classe_in_dic[i].d_name)):
                return classe_in_dic[i]
            else:
                continue
        return "no find"

In [ ]:
def find_concept_in_WN(identifiant_concept, dic_alphabet_WN):
    try:
        identifiant_concept[0].lower()
    except Exception:
        return "Not found"
    else:
        classe_in_dic = dic_alphabet_WN[identifiant_concept[0].lower()]
        for i in range(len(classe_in_dic)):
            if(op.eq(identifiant_concept, classe_in_dic[i].c_name)):
                return classe_in_dic[i]
            else:
                continue
        return "no find"

In [ ]:
#---------- Test
print(type(find_concept_in_MeSH('Calcimycin', dic_alphabet_MeSH)))
print(type(find_concept_in_WN('abattoir', dic_alphabet_WN)))

In [ ]:
identifiant_concept_MeSH = concept_a_verifie.iloc[1].Name_MeSH
print(identifiant_concept_MeSH)

In [ ]:
identifiant_concept_WN = concept_a_verifie.iloc[1].Name_WN
print(identifiant_concept_WN)

In [ ]:
print(dataConceptMeSH.iloc[1].values.tolist())

In [ ]:
#------------ find all concept of verifie in all concept WN and MeSH

In [ ]:
all_concept_find_MeSH = []
all_concept_find_WN = []
for i in range(1, concept_a_verifie.shape[0]):
    a = concept_a_verifie.iloc[i].Name_MeSH
    one_concept_MeSH = find_concept_in_MeSH(concept_a_verifie.iloc[i].Name_MeSH, dic_alphabet_MeSH)
    all_concept_find_MeSH.append(one_concept_MeSH.values.tolist())
    ont_concept_WN = find_concept_in_WN(concept_a_verifie.iloc[i].Name_WN, dic_alphabet_WN)
    all_concept_find_WN.append(ont_concept_WN.values.tolist())


In [ ]:
print(len(all_concept_find_MeSH))
print(len(all_concept_find_WN))

In [ ]:
print(all_concept_find_MeSH[1])
print(all_concept_find_WN[1])

In [ ]:
#---------turn list of list to dataframe
df_MeSH = pd.DataFrame(all_concept_find_MeSH, columns = ["c_ui", "d_name", "d_ui", "rel", "scopeNote", "tree", "word_list"])
df_WN = pd.DataFrame(all_concept_find_WN, columns = ["c_name", "synset", "hyponymes", "hypernyms", "definition", "exemple", "list_word"])
df_WN.head()

In [ ]:
#------------save csv
export_csv_WN = df_WN.to_csv(r'data_WN_searche_by_data_using_sim_sif_with_concept_a_verifie.csv', index = None, header = True)
export_csv_MeSH = df_MeSH.to_csv(r'data_MeSH_searche_by_data_using_sim_sif_with_concept_a_verifie.csv', index = None, header = True)


In [ ]:
#---------- load csv
df_WN = pd.read_csv('../Data_extracted/data_WN_searche_by_data_using_sim_sif_with_concept_a_verifie.csv', error_bad_lines= False, encoding='utf-8', names = ["c_name", "synset", "hyponymes", "hypernyms", "definition", "exemple", "list_word"])
df_MeSH = pd.read_csv('../Data_extracted/data_MeSH_searche_by_data_using_sim_sif_with_concept_a_verifie.csv', error_bad_lines= False, encoding='utf-8', names = ["c_ui", "d_name", "d_ui", "rel", "scopeNote", "tree", "word_list"])



In [ ]:
#---------find concept wn no hypo and store wn concept and mesh concept
synset = df_WN.iloc[1].synset
synset

In [ ]:
synset_clean = synset[8:-2]
print(synset_clean)

In [ ]:
hyper = wn.synset(synset_clean).hypernyms()
print(hyper[0])

In [ ]:
hypo = wn.synset(synset_clean).hyponyms()
print(hypo)

In [ ]:
def find_concept_wn_no_hypo(df_WN, df_MeSH):
    wn_concept_no_hypo = []
    wn_concept_has_hypo = []
    mesh_concept_no_hypo = []
    mesh_concept_has_hypo = []
    faux_synset = []
    for i in range(df_WN.shape[0]):
        synset = df_WN.iloc[i].synset
        synset_clean = synset[8:-2]
        try:
            list_hypo = wn.synset(synset_clean).hyponyms()
        except Exception:
            faux_synset.append(faux_synset)
        else:
            if(len(list_hypo) == 0):
                wn_concept_no_hypo.append(df_WN.iloc[i].values.tolist())
                mesh_concept_no_hypo.append(df_MeSH.iloc[i].values.tolist())
            else:
                wn_concept_has_hypo.append(df_WN.iloc[i].values.tolist())
                mesh_concept_has_hypo.append(df_MeSH.iloc[i].values.tolist())
    return (wn_concept_no_hypo, wn_concept_has_hypo, mesh_concept_no_hypo, mesh_concept_has_hypo)

In [ ]:
wn_concept_no_hypo, wn_concept_has_hypo, mesh_concept_no_hypo, mesh_concept_has_hypo = find_concept_wn_no_hypo(df_WN, df_MeSH)

In [ ]:
print(len(wn_concept_no_hypo))
print(len(wn_concept_has_hypo))
print(len(mesh_concept_no_hypo))
print(len(mesh_concept_has_hypo))

In [ ]:
df_MeSH_has_hypo = pd.DataFrame(mesh_concept_has_hypo, columns = ["c_ui", "d_name", "d_ui", "rel", "scopeNote", "tree", "word_list"])
df_MeSH_no_hypo = pd.DataFrame(mesh_concept_no_hypo, columns = ["c_ui", "d_name", "d_ui", "rel", "scopeNote", "tree", "word_list"])
df_WN_has_hypo = pd.DataFrame(wn_concept_has_hypo, columns = ["c_name", "synset", "hyponymes", "hypernyms", "definition", "exemple", "list_word"])
df_WN_no_hypo = pd.DataFrame(wn_concept_no_hypo, columns = ["c_name", "synset", "hyponymes", "hypernyms", "definition", "exemple", "list_word"])


In [ ]:
export_csv_WN_no_hypo = df_WN_no_hypo.to_csv(r'wn_no_hypo.csv', index = None, header = True)
export_csv_MeSH_no_hypo = df_MeSH_no_hypo.to_csv(r'mesh_no_hypo.csv', index = None, header = True)
export_csv_WN_has_hypo = df_WN_has_hypo.to_csv(r'wn_has_hypo.csv', index = None, header = True)
export_csv_MeSH_has_hypo = df_MeSH_has_hypo.to_csv(r'mesh_has_hypo.csv', index = None, header = True)


In [ ]:
#--------------Test
wn.synset('psychological_feature.n.01').hypernyms()

In [ ]:
wn.synset('abstraction.n.06').hypernyms()

In [ ]:
wn.synset('information.n.02').hypernyms()

In [ ]:
wn.synset('disease.n.01').hyponyms()

In [ ]:
wn.synset('apnea.n.01').hyponyms()

In [ ]:
#Extrat a tree number has the same length
def len_of_all_lever_tree_number():
    all_len_tree = []
    for i in range(1, dataConceptMeSH.shape[0]):
        getData = dataConceptMeSH['tree'][i].replace(' ', '')
        strToList = re.findall(r'[[](.*?)[]]', getData)[0].strip('[')
        target_list = [x for x in strToList.split(',')]
        #print(target_list)
        for j in range(len(target_list)):
            target_list[j] = target_list[j][1:-1]
            #print(target_list[j])
            #if(len(target_list[j] == 4)):
            #    print(target_list[j])
            all_len_tree.append(len(target_list[j]))
    return all_len_tree

def find_lever_of_tree():
    list_long_tree_number = len_of_all_lever_tree_number()
    list_remove_same_len = list(set(list_long_tree_number))
    list_remove_same_len.remove(4)#i dont know why len(tree_number) has 4
    lever_of_tree = sorted(list_remove_same_len, key=int, reverse=True)
    #inverse of Tree lever
    return lever_of_tree

In [ ]:
def find_a_lever_of_tree(lever_tree):
    lowest_list = []
    for i in range(1, dataConceptMeSH.shape[0]):
        getData = dataConceptMeSH['tree'][i].replace(' ', '')
        strToList = re.findall(r'[[](.*?)[]]', getData)[0].strip('[')
        target_list = [x for x in strToList.split(',')]
        for j in range(len(target_list)):
            des_lewest_every_lever = []
            target_list[j] = target_list[j][1:-1]
            if(len(target_list[j]) == lever_tree):
                lowest_list.append(target_list[j])
    return lowest_list

In [ ]:
def find_all_lever_of_tree():
    lever_tree = find_lever_of_tree()
    all_lever_class_by_lever_tree = []
    for i in lever_tree:
        all_lever_class_by_lever_tree.append(find_a_lever_of_tree(i))
    return all_lever_class_by_lever_tree

In [ ]:
all_lever_class = find_all_lever_of_tree()

In [ ]:
print(all_lever_class[12])

In [ ]:
dict_lever_tree = defaultdict(str)
dict_lever_tree = {key:[] for key in range(1, len(all_lever_class) + 1)}

In [ ]:
#df_MeSH_has_hypo
dict_lever_tree

In [ ]:
#------------- lever 1 is highest lever of tree.
all_lerver_class_filter = all_lever_class[::-1]

for i in range(len(all_lever_class)):
    dict_lever_tree[i+1] = all_lerver_class_filter[i]

In [ ]:
print(dict_lever_tree[2])

In [ ]:
def extract_tree_number_from_MeSH_has_hypo(concept_MeSH):
    tree_str = concept_MeSH.tree[1:-1]
    tree_str = tree_str.replace(' ','')
    list_tree = tree_str.split(',')
    list_tree = [tree[1:-1] for tree in list_tree]
    return list_tree

In [ ]:
ont_concept_MeSH = df_MeSH_has_hypo.iloc[1]
tree_number = extract_tree_number_from_MeSH_has_hypo(ont_concept_MeSH)
tree_number[0]

In [ ]:
def find_concept_by_tree_number(tree_number):
    concept_found = []
    for i in range(1, dataConceptMeSH.shape[0]):
        getData = dataConceptMeSH['tree'][i].replace(' ', '')
        strToList = re.findall(r'[[](.*?)[]]', getData)[0].strip('[')
        target_list = [x for x in strToList.split(',')]
        for j in range(len(target_list)):
            target_list[j] = target_list[j][1:-1]
            if(op.eq(tree_number, target_list[j])):
                concept_found.append(dataConceptMeSH.loc[i])
            else:
                pass
    return concept_found

In [ ]:
print(find_concept_by_tree_number(tree_number[0]))

In [ ]:
def find_previous_des(tree_number):
    return tree_number[0:-4]

In [ ]:
find_concept_by_tree_number(find_previous_des(tree_number[0]))

In [ ]:
synset = df_WN_has_hypo.iloc[1].synset
synset

In [ ]:
synset_clean = synset[8:-2]
synset_clean

In [ ]:
wn.synset(synset_clean).hypernyms()

In [ ]:
wn.synset('symptom.n.01').hypernyms()

In [ ]:
a = wn.synset('evidence.n.01').hypernyms()

In [ ]:
b = str(a[0]).split('.')

In [ ]:
b[0][8:]

In [ ]:
#----------Test similarity of deux identifiant concept WN and MeSH
fuzz.ratio("information", "INFORMATION")


In [ ]:
fuzz.partial_ratio("information", "INFORMATION")

In [ ]:
#-------------为什么我要写这个。。。。

def find_concept_MeSH_by_identifiant_WN_using_fuzz_partial_ratio(identifiant_wn, dic_alphabet_MeSH):
    try:
        identifiant_wn[0].lower()
    except Exception:
        return None
    else:
        classe = dic_alphabet_MeSH[identifiant_wn[0].lower()]
        concept_found = []
        for i in range(len(classe)):
            paire_fuzz = []
            socre = fuzz.partial_ratio(identifiant_wn, classe[i].d_name.lower())
            if(socre >= 85):
                paire_fuzz.append(classe[i])
                paire_fuzz.append(socre)
                concept_found.append(paire_fuzz)
                #return classe[i]
            else:
                continue
        return concept_found
    return None

In [ ]:
l = find_concept_MeSH_by_identifiant_WN_using_fuzz_partial_ratio('information', dic_alphabet_MeSH)

In [ ]:
print(l)

In [ ]:
#------------- Find father concept with concept has hypo WN and MeSH

In [ ]:
def find_concept_father_WN(synset_wn):
    synset_clean = synset_wn[8:-2]
    return wn.synset(synset_clean).hypernyms()

In [ ]:
syn1 = find_concept_father_WN("Synset('abdomen.n.01')")
syn2 = find_concept_father_WN("Synset('abscess.n.01')")
print(syn2)

In [ ]:
def find_concept_father_MeSH(tree_numer_MeSH):
    return find_concept_by_tree_number(find_previous_des(tree_numer_MeSH))

In [ ]:
find_concept_father_MeSH('A01.923.047')
find_concept_father_MeSH('C01.539.830.025')

In [ ]:
def take_concept_voisin(df_MeSH_has_hypo, df_WN_has_hypo):
    for i in range(df_MeSH_has_hypo.shape[0]):
        synset_father = find_concept_father_WN(df_WN_has_hypo.iloc[i].synset)
        print(synset_father)
        list_tree = extract_tree_number_from_MeSH_has_hypo(df_MeSH_has_hypo.iloc[i])
        for j in range(len(list_tree)):
            all_father = []
            one_father_MeSh = find_concept_father_MeSH(list_tree[j])
            all_father.append(one_father_MeSh)
            print(all_father)                                     
                                               

In [ ]:
take_concept_voisin(df_MeSH_has_hypo, df_WN_has_hypo)

In [ ]:
#---------------create a graphe
G = nx.MultiDiGraph()

In [ ]:
G.add_node(synset_clean)
G.add_node(hyper[0])
print(len(hypo))
G.add_node(hypo[0])

In [ ]:
G.has_node(synset_clean)

In [ ]:
G.add_edge(hyper[0], synset_clean)
G.add_edge(synset_clean, hypo[0])

In [ ]:
nx.draw(G, with_labels=True, font_weight='bold')

In [ ]:
#dict_lever_tree

In [ ]:
def find_concept_hyponyme_in_MeSH_by_using_tree_number(tree_number):
    # lever of this tree number
    index = len(tree_number)//3
    all_concept_hypoynme_of_this_tree_number = []
    #index + 1 = begin from next lever tree
    #index + 2 = end from next next lever tree
    for i in range(index + 1, index + 2):
        for j in range(len(dict_lever_tree[i])):
            if(fuzz.partial_ratio(dict_lever_tree[i][j],tree_number) == 100):
                all_concept_hypoynme_of_this_tree_number.append(find_concept_by_tree_number(dict_lever_tree[i][j]))
            else:
                continue
    return all_concept_hypoynme_of_this_tree_number

In [ ]:
tree_number = 'A01.923.047'
tree_number_next = 'A01.923.047.031'
l_hyponyme = find_concept_hyponyme_in_MeSH_by_using_tree_number(tree_number)

In [ ]:
find_concept_by_tree_number(tree_number)
print(l_hyponyme)

In [ ]:
def find_father_concept_WN_by_using_identifiant_of_father_concept_MeSH(i_father_mesh, child_synset_wn):
    l_hyper = wn.synset(child_synset_wn).hypernyms()
    #图是方便看所以创建的可以删掉所有有关图的信息这个方法还是可以运行
    G_WN.add_node(child_synset_wn)
    for father_synset in l_hyper:
        #print(father_synset)
        to_string = str(father_synset)
        father_synset_clear = to_string[8:-2]
        l_synset_father = father_synset_clear.split('.')
        #图是方便看所以创建的可以删掉所有有关图的信息这个方法还是可以运行
        ratio = fuzz.partial_ratio(l_synset_father[0], i_father_mesh)
        G_WN.add_node(father_synset_clear)
        G_WN.add_edge(father_synset_clear, child_synset_wn)
        #print(father_synset_clear)
        #print(ratio)
        #print(l_synset_father[0])
        if(fuzz.partial_ratio(l_synset_father[0], i_father_mesh) >= 80):
            return (father_synset_clear, ratio)
        else:
            #------------ 没有return的时候数据只能进不能出 所以在else也要用return
            return find_father_concept_WN_by_using_identifiant_of_father_concept_MeSH(i_father_mesh, father_synset_clear)

In [ ]:
def check_equal_identify_concept(list_identify_concept):
    return op.eq(list_identify_concept[1:], list_identify_concept[:-1])

In [ ]:
#-------- Test
lst_1 = ['aze', 'aze', 'aze']
lst_2 = ['aze', 'az', 'aze']
lst_3 = []
print(check_equal_identify_concept(lst_1))
print(check_equal_identify_concept(lst_2))
print(check_equal_identify_concept(lst_3))

In [ ]:
def find_father_concept_MeSH_if_concept_has_different_father(list_concept_tree_number):
    if (len(list_concept_tree_number) >= 2):
        all_father_name = []
        for tree_number in list_concept_tree_number:
            concept_father = find_concept_by_tree_number(find_previous_des(tree_number))
            all_father_name.append(concept_father[0].d_name)
        if(check_equal_identify_concept(all_father_name)):
            return find_father_concept_MeSH_if_concept_has_different_father(extract_tree_number_from_MeSH_has_hypo(concept_father[0]))
        else:
            #eles concept father is different, we returen a list of concept of those father.
            l_concept_father = []
            for i in range(len(all_father_name)):
                l_concept_father.append(find_concept_in_MeSH(all_father_name[i], dic_alphabet_MeSH))
            return l_concept_father
    else:
        #if it has only one father, we return directely his father concept.
        return find_concept_by_tree_number(list_concept_tree_number[0])

In [ ]:
tree_number = extract_tree_number_from_MeSH_has_hypo(df_MeSH_has_hypo.iloc[1])
print(tree_number)
l = find_father_concept_MeSH_if_concept_has_different_father(tree_number)
print(l)

In [ ]:
print(tree_number)
concept = find_concept_by_tree_number(find_previous_des(tree_number[0]))
print(concept)
#print(df_MeSH_has_hypo.iloc[1])
#find_father_concept_MeSH_if_concept_has_different_father()


In [ ]:
wn.synset('vegetarian.n.01').hyponyms()

In [ ]:
G_WN = nx.MultiDiGraph()
reponse = find_father_concept_WN_by_using_identifiant_of_father_concept_MeSH('Persons', 'vegetarian.n.01')
nx.draw(G_WN, with_labels=True, font_weight='bold')

In [ ]:
list(G_WN.edges())

In [ ]:
print(reponse + ('123',))

In [ ]:
G_WN = nx.MultiDiGraph()
reponse = find_father_concept_WN_by_using_identifiant_of_father_concept_MeSH('Torso', 'abdomen.n.01')
nx.draw(G_WN, with_labels=True, font_weight='bold')

In [ ]:
reponse = find_father_concept_WN_by_using_identifiant_of_father_concept_MeSH('Fabaceae', 'acacia.n.01')

In [ ]:
print(type(reponse))

In [ ]:
reponse = find_father_concept_WN_by_using_identifiant_of_father_concept_MeSH('Publication Formats', 'calendar.n.02')

In [ ]:
reponse = find_father_concept_WN_by_using_identifiant_of_father_concept_MeSH('Emotions', 'disgust.n.01')

In [ ]:
G_WN = nx.MultiDiGraph()
reponse = find_father_concept_WN_by_using_identifiant_of_father_concept_MeSH('Alcohol-Related Disorders', 'alcohol.n.01')
pos = nx.spring_layout(G_WN, k=1, iterations=100)
nx.draw(G_WN, pos, with_labels=True, font_weight='bold')

In [ ]:
#------------- Save graphe to xml
nx.write_graphml(G_WN, 'test.graphhml')

In [ ]:
#------------- Read graphe
a = nx.read_graphml('test.graphhml')

In [ ]:
a

In [ ]:
nx.draw(a, with_labels=True, font_weight='bold')

In [ ]:
l_tree_number = extract_tree_number_from_MeSH_has_hypo(df_MeSH_has_hypo.iloc[1])
father_concept = find_concept_by_tree_number(find_previous_des(l_tree_number[0]))


In [ ]:
father_concept[0].d_name

In [ ]:
def algo_comparaison_voisin():
    l_mesh_chiled_father = []
    no_father_mesh = []
    for i in range(1, df_MeSH_has_hypo.shape[0]):
        #--------create a graph for mesh and
        G_WN = nx.MultiDiGraph()
        #----- principe
        chile_name = df_MeSH_has_hypo.iloc[i].d_name
        l_tree_number = extract_tree_number_from_MeSH_has_hypo(df_MeSH_has_hypo.iloc[i])
        # 新加的
        l_father_concept = find_father_concept_MeSH_if_concept_has_different_father(l_tree_number)
        for i in range(len(l_father_concept)):
            father_concept = find_concept_by_tree_number(find_previous_des(l_tree_number[i]))
            try:
                identifie_mesh = father_concept[0].d_name
            except Exception:
                no_father_mesh.append(father_concept)
                continue
            else:
                l_mesh_chiled_father.append((father_concept[0].d_name, df_MeSH_has_hypo.iloc[i].d_name))
                synset_chiled = (df_WN_has_hypo.iloc[i].synset)[8:-2]
                result = find_father_concept_WN_by_using_identifiant_of_father_concept_MeSH(identifie_mesh, synset_chiled)
                #----------- 如果找到了相同的就返回
                if (type(result) == tuple):
                    result1  = result + (synset_chiled, )
                    result2 = result1 + (identifie_mesh, )
                    result3 = result2 + (chile_name, )
                    l_mesh_chiled_father.append(result3)
                else:
                    continue
        #----- graph
        #G_WN.add_node(father_concept[0].d_name)
        #G_WN.add_node(df_MeSH_has_hypo.iloc[i].d_name)
        #G_WN.add_edge(father_concept[0].d_name, df_MeSH_has_hypo.iloc[i].d_name)
        #graphe_paire = nx.draw(G_WN, with_labels=True, font_weight='bold')
        #print(graphe_paire)
        #nx.write_graphml(G_WN, ('./concept_mesh/' + df_MeSH_has_hypo.iloc[i].d_name + '.graphhml'))
    return l_mesh_chiled_father, no_father_mesh

In [ ]:
l_mesh, l_no_father = algo_comparaison_voisin()

In [ ]:
l_fusionable = []
for i in range(len(l)):
    if(len(l[i]) == 5):
        l_fusionable.append(l[i])

In [ ]:
print(l_fusionable)

In [ ]:
df_fusionable = pd.DataFrame(l_fusionable, columns = ['synset_father_wn', 'Parti_Ratio(fuzz)_between_father_concept', 'synset_child_wn', 'identify_father_MeSH', 'identify_child_MesH'])

In [ ]:
df_fusionable.head()

In [ ]:
export = df_fusionable.to_csv(r'concept_fusonable_after_second_algo_for_concept_has_hypo.csv', index = None, header = True)

In [ ]:
(df_WN_has_hypo.iloc[2].synset)[8:-2]

In [ ]:
print(l_mesh_chiled_father)

In [ ]:
#for i in range(len(l_mesh_chiled_father)):
G_WN = nx.MultiDiGraph()
G_WN.add_edges_from(l_mesh_chiled_father)
nx.draw(G_WN, with_labels=True, font_weight='bold')